# Capstone 3 — Structured Extraction Pipeline at Scale · Homework (self-test)

**Goal:** check that you can design an extraction contract that does not invite fabrication, validate and retry the right things, resubmit only failed batch items with the right fix, plan an SLA, price a run honestly, and calibrate review routing.

- Brief and theory pointers: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 90–120 minutes
- Estimated API cost: **$0** (everything is offline and graded locally)
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 10 scenario questions in the `answers` cell, then run the grader. The questions are original; they test the same competencies as the exam's extraction scenario but are not exam items.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises), the same bar as the Claude exam's 720/1000.

The exercises use a second domain on purpose (insurance claim intake) so you transfer the patterns instead of copying the practice code.

In [ ]:
import copy
import hashlib
import json
import os
import re
from collections import defaultdict
from datetime import date
from typing import Callable, Literal, Optional

from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, ConfigDict, ValidationError

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
print("MODEL =", MODEL, "(only used to build request dicts; nothing is sent)")

### Helpers (run, don't edit)

In [ ]:
# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {exc}")


# ---------- the insurance-claim contract used by the exercises ----------
CONFIDENCE_FIELDS = ("claim_type", "policy_number", "incident_date", "claimed_amount_cents")
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
TRANSIENT = {"overloaded_error", "api_error", "server_error"}


class ClaimItem(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    description: str
    amount_cents: int


class ClaimExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    claim_type: Literal["water_damage", "fire", "theft", "other"]
    claim_type_detail: Optional[str]
    policy_number: Optional[str]
    incident_date: Optional[str]
    claimed_amount_cents: Optional[int]
    police_report_ref: Optional[str] = None
    items: list[ClaimItem]
    conflict_detected: bool
    field_confidence: dict[str, float]


# Exercise 3 references this name. It is a placeholder so Exercise 3 does not depend on Exercise 1;
# once Exercise 1 passes you may set CLAIM_SCHEMA_CLAUDE = build_claim_schemas()[0].
CLAIM_SCHEMA_CLAUDE = {"type": "object", "description": "placeholder: Exercise 1 builds the real schema"}


def split_lines(text: str, max_chars: int) -> list[str]:
    """Split on line boundaries into chunks of at most max_chars (a single longer line becomes its own chunk)."""
    chunks, current = [], ""
    for line in text.splitlines():
        candidate = f"{current}\n{line}" if current else line
        if current and len(candidate) > max_chars:
            chunks.append(current)
            candidate = line
        current = candidate
    return chunks + ([current] if current else [])


def good_claim(**overrides) -> dict:
    """A valid extraction; the checkers derive their cases from it."""
    base = {"claim_type": "water_damage", "claim_type_detail": None, "policy_number": "HP-204-7781",
            "incident_date": "2026-08-14", "claimed_amount_cents": 182_500, "items": [
                {"description": "Flooring replacement", "amount_cents": 150_000},
                {"description": "Dehumidifier rental", "amount_cents": 32_500}],
            "conflict_detected": False,
            "field_confidence": {"claim_type": 0.97, "policy_number": 0.95, "incident_date": 0.9, "claimed_amount_cents": 0.93}}
    base.update(overrides)
    return base


print("helpers ready")

## Part A — Quiz (10 questions)

Pick one letter per question. Domains: 4.2–4.5 (few-shot, schemas, validation, batches) and 5.5 (calibration and review), plus the OpenAI Batch API.

**Q1. (Schema design)** A pipeline extracts supplier safety certificates. `expiry_date` is a required string in the extraction schema. About 18% of certificates have no expiry date, and on those the model returns a date exactly one year after the issue date. What is the most effective fix?

- A) Add "Do not guess expiry dates" to the system prompt.
- B) Keep the key required but allow `null`, describe `null` as "no expiry date is printed", and tell the model to use it when the value is absent.
- C) Post-process the output and delete any expiry date that is exactly one year after the issue date.
- D) Retry each certificate with feedback saying "expiry_date looks fabricated".

**Q2. (Categories)** A facilities team classifies work orders into a 6-value `category` enum. New kinds of work keep arriving, and analysts notice that "maintenance" has quietly become a catch-all holding a third of all orders. What should the schema change be?

- A) Expand the enum to 40 values covering every case seen so far.
- B) Replace the enum with a free-text `category` string.
- C) Add `"other"` with a `category_detail` string, and periodically review the detail values to promote frequent ones into the enum.
- D) Add a confidence score to `category` and auto-accept only scores above 0.9.

**Q3. (Retry limits)** Validation shows `vat_number` is `null` on 6% of supplier onboarding forms. The team proposes three retries with the feedback "vat_number is required and must not be null". Looking at the forms, all of them come from sole traders who are not VAT-registered. What should you do?

- A) Keep the three retries; persistence usually recovers missing fields.
- B) Retry the same forms on a larger model.
- C) Add few-shot examples that show VAT numbers being extracted from forms.
- D) Treat `null` as a valid answer for this segment, stop retrying, and route those forms to review only if the business process needs a VAT number.

**Q4. (Syntax vs semantics)** After moving to strict structured outputs, JSON parse errors drop to zero, but 3% of extracted invoices have line items that do not add up to the total. What is the right next step?

- A) Compute the sum in code and compare it with the printed total; on a mismatch, retry with the specific discrepancy, and have the model set a conflict flag when the document itself is inconsistent.
- B) Report a bug: strict mode is supposed to prevent inconsistent values.
- C) Instruct the model to adjust the line items so they always match the total.
- D) Increase `max_tokens` so the model has room to finish its arithmetic.

**Q5. (`tool_choice`)** A pipeline receives invoices, receipts and contracts in one queue, and the type is not known in advance. There is one strict extraction tool per type. The model in use accepts forced tool use. You need a structured extraction on every request, chosen by the model. Which setting?

- A) `{"type": "auto"}`
- B) `{"type": "any"}`
- C) `{"type": "tool", "name": "extract_invoice"}`
- D) `{"type": "none"}`

**Q6. (Batch failures)** A batch of 2,000 documents ends with 1,940 usable results, 35 `expired`, 15 `errored` with "prompt is too long", and 10 `succeeded` with `stop_reason: "max_tokens"`. What should the next batch contain?

- A) All 60 failed requests, unchanged.
- B) The whole batch again, because more than 1% failed.
- C) The 35 expired requests unchanged, the 15 long documents split into chunks under new `custom_id`s with a chunk suffix, and the 10 truncated requests with a higher `max_tokens`.
- D) Only the 35 expired requests; the other 25 go straight to human review.

**Q7. (SLA)** Documents arrive around the clock and must be extracted within 40 hours of arrival. Plan with 24 hours of worst-case batch processing and 2 hours for validation and synchronous retries. What is the longest interval between batch submissions that still meets the SLA?

- A) 1 hour, because most batches finish within an hour.
- B) 14 hours.
- C) 16 hours.
- D) 40 hours, because the SLA starts at submission.

**Q8. (Calibration)** The model's self-reported confidence on `iban` averages 0.92. The team wants to auto-accept every `iban` with confidence ≥ 0.9 starting next week. What should happen first?

- A) Approve it: a 0.9 score means a 90% chance of being correct, which beats manual keying.
- B) Ask the model to be more conservative with its scores.
- C) Run each document twice and auto-accept when both runs agree.
- D) Build a reliability table on a labeled validation set, per document type, and pick the threshold that meets the target accuracy with enough samples.

**Q9. (Monitoring auto-accepted output)** After calibration, 92% of documents are auto-accepted and only low-confidence fields reach reviewers. How will the team notice a new error pattern in auto-accepted documents, for example after a major supplier changes its invoice layout?

- A) Review a random sample of auto-accepted documents, stratified by document type (and confidence band), with a minimum per stratum.
- B) Rely on reviewers: errors will show up in the low-confidence queue.
- C) Audit the 1% of auto-accepted documents with the lowest confidence.
- D) Wait for supplier complaints about wrong payments.

**Q10. (OpenAI Batch API)** To save cost, a team writes simple documents for `gpt-6-luna` and complex ones for `gpt-6.1-sol` into a single JSONL input file for the Batch API. A reviewer blocks the upload because the file breaks a documented Batch API rule. Which change makes the job valid?

- A) Pass the model in `batches.create(...)` instead of in each line's `body`.
- B) Use the `/v1/chat/completions` endpoint, which allows mixed models.
- C) Split the work into one input file and one batch per model.
- D) Prefix each `custom_id` with the model name.

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_HASHES = {
    "q1": "96e2e2c94d02b534d4bdc303373df7b704ed4725981fac17549ee985d132c365",
    "q2": "c85dcf1566dc5c0487a6fe1611c41ba32751e8080b552a419521a39c7fee21b2",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "0eca085218afe99ae6fb326d50c6e730cd71c102ccfe2f9a10d8e6054e2018e0",
    "q5": "fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c",
    "q6": "18cb69e7d9c041e4c1b21fa8c0e110716664b55bef1348d49301f8891c82cef1",
    "q7": "16dd052a869d89a87df9ae09526a5477a3e6870575379b0c9d2ed35de3ab21c7",
    "q8": "85b91a1895880cedf03bb98aa9ebfb3c730c838ec5d8f59ec2d1d86bcb936ced",
    "q9": "8b77cfcdcf7bee660b161787fd482af13fe960d0da61692597e6e870d4668936",
    "q10": "d6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5"
}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — Claim schemas for both vendors' strict modes

Write `build_claim_schemas()` returning `(claude_schema, openai_schema)` for this contract:

| Field | Type | Notes |
|---|---|---|
| `claim_type` | string enum, **not** nullable | `water_damage`, `fire`, `theft`, plus the extensible bucket |
| `claim_type_detail` | string or null | What the claim is when the bucket is used |
| `policy_number`, `incident_date` | string or null | May be absent from the form |
| `claimed_amount_cents` | integer or null | May be absent (illegible) |
| `police_report_ref` | string | **Optional**: Claude leaves it out of `required`; OpenAI strict needs every key required, so make it nullable there |
| `items` | array of objects `{description: string, amount_cents: integer}` | Both keys required |
| `conflict_detected` | boolean | |
| `field_confidence` | object with one `number` per name in `CONFIDENCE_FIELDS` | All required |

Rules: every object (root, items, `field_confidence`) has `additionalProperties: false`; every key except Claude's `police_report_ref` is in `required`; express "nullable" as `["<type>", "null"]` or `anyOf` with `{"type": "null"}`; do not use `minimum`/`maximum` (Claude's strict mode rejects them; check ranges in code).

Theory: [README 3.2](README.md#32-extraction-schema-design)

In [ ]:
def build_claim_schemas() -> tuple[dict, dict]:
    """Return (claude_schema, openai_schema). Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def _nullable(p: dict) -> bool:
    t = p.get("type")
    return (isinstance(t, list) and "null" in t) or any(o.get("type") == "null" for o in p.get("anyOf", []))


def _types(p: dict) -> set:
    """The non-null JSON types a property allows (from "type" or from anyOf branches)."""
    out = set()
    for branch in [p] + p.get("anyOf", []):
        t = branch.get("type")
        out |= set(t) if isinstance(t, list) else ({t} if t else set())
    return out - {"null"}


def _objects(s):
    if isinstance(s, dict):
        if s.get("type") == "object" or (isinstance(s.get("type"), list) and "object" in s["type"]):
            yield s
        for v in s.values():
            yield from _objects(v) if isinstance(v, (dict, list)) else ()
    elif isinstance(s, list):
        for v in s:
            yield from _objects(v)


EX1_KEYS = {"claim_type", "claim_type_detail", "policy_number", "incident_date", "claimed_amount_cents",
            "police_report_ref", "items", "conflict_detected", "field_confidence"}


def check_ex1() -> None:
    def check():
        out = build_claim_schemas()
        assert isinstance(out, tuple) and len(out) == 2, "return a tuple (claude_schema, openai_schema)"
        for name, s in zip(("claude", "openai"), out):
            assert isinstance(s, dict) and s.get("type") == "object", f"{name}: the root must be a JSON object schema"
            props = s.get("properties", {})
            assert set(props) == EX1_KEYS, f"{name}: properties must be exactly {sorted(EX1_KEYS)}; got {sorted(props)}"
            for obj in _objects(s):
                assert obj.get("additionalProperties") is False, \
                    f"{name}: every object needs additionalProperties: false (check {sorted(obj.get('properties', {}))})"
            want = {"claim_type": "string", "claim_type_detail": "string", "policy_number": "string", "incident_date": "string",
                    "claimed_amount_cents": "integer", "police_report_ref": "string", "items": "array",
                    "conflict_detected": "boolean", "field_confidence": "object"}
            for f, t in want.items():
                assert _types(props[f]) == {t}, f"{name}: {f} must be of type {t!r} (plus null where the table says so); got {sorted(_types(props[f]))}"
            ct = props["claim_type"]
            assert not _nullable(ct), f"{name}: claim_type is always present: do not make it nullable"
            enum = ct.get("enum") or []
            assert {"water_damage", "fire", "theft"} <= set(enum), f"{name}: claim_type enum must list the three known types"
            assert "other" in enum, f"{name}: a closed enum forces new claim kinds into a wrong bucket: add 'other' (+ the detail field)"
            assert _nullable(props["claim_type_detail"]), f"{name}: claim_type_detail must be nullable (null unless claim_type is 'other')"
            for f in ("policy_number", "incident_date", "claimed_amount_cents"):
                assert _nullable(props[f]), (f"{name}: {f} may be absent from the form. A non-nullable field pushes the model "
                                             "to INVENT a value: make it nullable")
            items = props["items"]
            assert items.get("type") == "array" and items.get("items", {}).get("type") == "object", f"{name}: items is an array of objects"
            ip = items["items"].get("properties", {})
            assert ip.get("amount_cents", {}).get("type") == "integer" and ip.get("description", {}).get("type") == "string", \
                f"{name}: item properties are description (string) and amount_cents (integer)"
            assert set(items["items"].get("required", [])) == {"description", "amount_cents"}, f"{name}: both item keys are required"
            assert props["conflict_detected"].get("type") == "boolean", f"{name}: conflict_detected is a boolean"
            fc = props["field_confidence"]
            assert set(fc.get("properties", {})) == set(CONFIDENCE_FIELDS) and \
                all(v.get("type") == "number" for v in fc["properties"].values()), \
                f"{name}: field_confidence has one number per CONFIDENCE_FIELDS entry"
            assert set(fc.get("required", [])) == set(CONFIDENCE_FIELDS), f"{name}: every confidence key is required"
        claude, openai = out
        assert "minimum" not in json.dumps(claude) and "maximum" not in json.dumps(claude), \
            "claude: strict schemas reject minimum/maximum; validate the 0-1 range in code instead"
        assert set(claude.get("required", [])) == EX1_KEYS - {"police_report_ref"}, \
            ("claude: everything except police_report_ref is required; police_report_ref is OPTIONAL "
             "(left out of required). Claude strict mode allows optional keys")
        assert set(openai.get("required", [])) == EX1_KEYS, \
            "openai: strict mode rejects optional keys: list EVERY property in required (use null for 'absent')"
        assert _nullable(openai["properties"]["police_report_ref"]), \
            "openai: police_report_ref is required there, so it must be nullable to express 'not on the form'"
    _run_check(1, check)


check_ex1()

### Exercise 2 — A semantic validator with retryable flags

Write `validate_claim(data, attempt=0)` returning a list of `(field, code, retryable)` tuples (order does not matter; `[]` = valid):

| Situation | Tuple |
|---|---|
| `ClaimExtraction.model_validate(data)` raises | **only** `("root", "schema", True)` |
| `incident_date` is not `YYYY-MM-DD`, or is not a real calendar date | `("incident_date", "date_format", True)` |
| `claim_type == "other"` and `claim_type_detail` is empty or `None` | `("claim_type_detail", "other_without_detail", True)` |
| any `field_confidence` value outside 0–1 | `("field_confidence", "confidence_range", True)` |
| `claimed_amount_cents is None` | `("claimed_amount_cents", "missing_amount", attempt == 0)`: one honest re-read, then a human |
| amount present, `sum(items) != claimed_amount_cents`, and `conflict_detected` is false | `("items", "sum_mismatch", True)` |

A mismatch with `conflict_detected: true` is **not** a validation issue: the source itself is inconsistent, and that goes to a person, not to a retry.

Theory: [README 3.4](README.md#34-validation-and-retry-with-error-feedback)

In [ ]:
def validate_claim(data: dict, attempt: int = 0) -> list[tuple[str, str, bool]]:
    """Return (field, code, retryable) tuples. Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        def got(data, attempt=0):
            out = validate_claim(data, attempt)
            assert isinstance(out, list), f"return a list of tuples, got {type(out).__name__}"
            return {tuple(x) for x in out}

        assert got(good_claim()) == set(), f"a valid claim must return []; got {got(good_claim())}"
        cases = [
            (good_claim(claim_type="flood"), 0, {("root", "schema", True)}, "an enum violation is a schema error; return ONLY the schema tuple"),
            (good_claim(claimed_amount_cents="1825.00"), 0, {("root", "schema", True)}, "a string amount is a schema error (strict types)"),
            (good_claim(incident_date="14/08/2026"), 0, {("incident_date", "date_format", True)}, "dates must be YYYY-MM-DD"),
            (good_claim(incident_date="2026-02-30"), 0, {("incident_date", "date_format", True)},
             "2026-02-30 matches the pattern but is not a real date: check it with date.fromisoformat"),
            (good_claim(incident_date=None), 0, set(), "a null incident_date is an honest 'not on the form', not an error"),
            (good_claim(claim_type="other", claim_type_detail=None), 0, {("claim_type_detail", "other_without_detail", True)},
             "'other' needs its detail"),
            (good_claim(claim_type="other", claim_type_detail="hail damage to roof"), 0, set(), "'other' WITH a detail is valid"),
            (good_claim(field_confidence={"claim_type": 97, "policy_number": 0.9, "incident_date": 0.9, "claimed_amount_cents": 0.9}), 0,
             {("field_confidence", "confidence_range", True)}, "97 is outside 0-1 (a percentage slipped in)"),
            (good_claim(claimed_amount_cents=None), 0, {("claimed_amount_cents", "missing_amount", True)},
             "first attempt: one re-read is worth trying"),
            (good_claim(claimed_amount_cents=None), 1, {("claimed_amount_cents", "missing_amount", False)},
             "after one retry, a still-missing amount is probably absent from the form: not retryable"),
            (good_claim(claimed_amount_cents=190_000), 0, {("items", "sum_mismatch", True)}, "items sum to 182,500, not 190,000"),
            (good_claim(claimed_amount_cents=190_000, conflict_detected=True), 0, set(),
             "conflict_detected=True: the source is inconsistent. That is routed to a person, not retried"),
            (good_claim(incident_date="Aug 14", claim_type="other", claim_type_detail="", claimed_amount_cents=1), 0,
             {("incident_date", "date_format", True), ("claim_type_detail", "other_without_detail", True), ("items", "sum_mismatch", True)},
             "report EVERY issue at once, so one retry can fix them all"),
        ]
        for data, attempt, expected, why in cases:
            result = got(data, attempt)
            assert result == expected, f"{why}. Expected {sorted(expected)}, got {sorted(result)}"
    _run_check(2, check)


check_ex2()

### Exercise 3 — The retry request and its stopping rule

Write `build_retry_request(custom_id, document, failed, issues, attempt, max_attempts=2)`. `issues` uses Exercise 2's tuple format.

- Return `None` when no issue is retryable, or when `attempt + 1 >= max_attempts` (the retry budget is spent).
- Otherwise return a Claude batch request dict `{"custom_id": custom_id, "params": {...}}` with `model=MODEL`, `max_tokens=1024`, `output_config={"format": {"type": "json_schema", "schema": CLAIM_SCHEMA_CLAUDE}}`, and three messages: the **document** (user), the **failed extraction** as JSON text (assistant), and the **feedback** (user).
- The feedback lists every **retryable** issue as `- <field>: <code>` on its own line, does **not** mention non-retryable issues, and tells the model to keep a value `null` if it is not in the document. Never demand a value ("must not be null").

Theory: [README 3.4](README.md#34-validation-and-retry-with-error-feedback)

In [ ]:
def build_retry_request(custom_id: str, document: str, failed: dict, issues: list[tuple[str, str, bool]],
                        attempt: int, max_attempts: int = 2) -> dict | None:
    """Return a Claude batch request for the retry, or None. Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        doc = "<document id=\"clm-0042\">\nClaim form ... water damage ... 14/08/2026 ...\n</document>"
        failed = good_claim(incident_date="14/08/2026", claimed_amount_cents=None)
        issues = [("incident_date", "date_format", True), ("claimed_amount_cents", "missing_amount", False)]
        assert build_retry_request("clm-0042", doc, failed, [("claimed_amount_cents", "missing_amount", False)], 0) is None, \
            "no retryable issue: return None (a retry cannot add information that is not on the form)"
        assert build_retry_request("clm-0042", doc, failed, issues, attempt=1, max_attempts=2) is None, \
            "attempt 1 of max 2: the retry budget is spent, return None (route to a person)"
        req = build_retry_request("clm-0042", doc, failed, issues, attempt=0)
        assert isinstance(req, dict) and req.get("custom_id") == "clm-0042", "keep the SAME custom_id: one logical item = one key"
        p = req.get("params", {})
        assert p.get("model") == MODEL and p.get("max_tokens") == 1024, "params need model=MODEL and max_tokens=1024"
        assert "stream" not in p, "batch requests cannot stream"
        assert p.get("output_config", {}).get("format", {}).get("schema") == CLAIM_SCHEMA_CLAUDE and \
            p["output_config"]["format"].get("type") == "json_schema", "use output_config.format with type json_schema and CLAIM_SCHEMA_CLAUDE"
        msgs = p.get("messages", [])
        assert [m.get("role") for m in msgs] == ["user", "assistant", "user"], \
            "three turns: user (document), assistant (failed extraction), user (feedback)"
        assert msgs[0]["content"] == doc, "resend the ORIGINAL document: the model must re-read the source"
        assert json.loads(msgs[1]["content"]) == failed, "the assistant turn is the failed extraction as JSON text"
        fb = msgs[2]["content"]
        assert "- incident_date: date_format" in fb, "list each retryable issue as '- <field>: <code>'"
        assert "missing_amount" not in fb and "claimed_amount_cents" not in fb, \
            "do not send non-retryable issues: asking again for absent information invites fabrication"
        assert "null" in fb.lower() and "must not be null" not in fb.lower(), \
            "tell the model to keep values null when they are not in the document (and never demand a value)"
        two = build_retry_request("clm-7", doc, failed, [("incident_date", "date_format", True), ("items", "sum_mismatch", True)], 0)
        fb2 = two["params"]["messages"][2]["content"]
        assert "- incident_date: date_format" in fb2 and "- items: sum_mismatch" in fb2, "include EVERY retryable issue"
    _run_check(3, check)


check_ex3()

### Exercise 4 — Resubmit only the failures, each with the right fix

Write `plan_resubmission(outcomes, requests, max_chunk_chars=2_000)`. `outcomes` are normalized results (`{"custom_id", "status", "error_type", "error_message"}`, status one of `ok`, `expired`, `canceled`, `errored`, `truncated`, `refused`); `requests` maps each `custom_id` to `{"document": str, "max_tokens": int}`. Return `(resubmit, human)`:

| Outcome | Action |
|---|---|
| `ok` | nothing |
| `expired`, `canceled`, or `errored` with `error_type` in `TRANSIENT` | resubmit **unchanged** |
| `errored` whose type or message contains `"prompt is too long"` (Claude's documented message) or `"context_length_exceeded"` (the label the practice notebook's OpenAI fake uses; check what your real runs return) | split with `split_lines(document, max_chunk_chars)` into `{custom_id}-c0`, `-c1`, ... (same `max_tokens`) |
| `truncated` | resubmit with `max_tokens` doubled |
| anything else (`refused`, other errors) | `human` (a list of `custom_id`s) |

`resubmit` is a list of `{"custom_id", "document", "max_tokens"}` dicts.

Theory: [README 3.5](README.md#35-batch-processing-on-both-stacks)

In [ ]:
def plan_resubmission(outcomes: list[dict], requests: dict[str, dict],
                      max_chunk_chars: int = 2_000) -> tuple[list[dict], list[str]]:
    """Return (resubmit, human). Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        long_doc = "\n".join(f"line {i:04d} of a very long annex with delivery records" for i in range(150))
        reqs = {cid: {"document": f"<document id=\"{cid}\">text {cid}</document>", "max_tokens": 1024}
                for cid in ("ok-1", "exp-1", "can-1", "ovl-1", "srv-1", "long-1", "long-2", "trunc-1", "ref-1", "bad-1")}
        reqs["long-1"]["document"] = long_doc
        reqs["long-2"]["document"] = long_doc
        outcomes = [
            {"custom_id": "ok-1", "status": "ok", "error_type": None, "error_message": None},
            {"custom_id": "exp-1", "status": "expired", "error_type": None, "error_message": None},
            {"custom_id": "can-1", "status": "canceled", "error_type": None, "error_message": None},
            {"custom_id": "ovl-1", "status": "errored", "error_type": "overloaded_error", "error_message": "Overloaded"},
            {"custom_id": "srv-1", "status": "errored", "error_type": "server_error", "error_message": "500"},
            {"custom_id": "long-1", "status": "errored", "error_type": "invalid_request_error",
             "error_message": "prompt is too long: 9120 tokens > 4000 maximum"},
            {"custom_id": "long-2", "status": "errored", "error_type": "context_length_exceeded", "error_message": "Input too long."},
            {"custom_id": "trunc-1", "status": "truncated", "error_type": None, "error_message": None},
            {"custom_id": "ref-1", "status": "refused", "error_type": None, "error_message": None},
            {"custom_id": "bad-1", "status": "errored", "error_type": "invalid_request_error",
             "error_message": "tool_choice tool is not supported on this model"},
        ]
        out = plan_resubmission(outcomes, copy.deepcopy(reqs))
        assert isinstance(out, tuple) and len(out) == 2, "return a tuple (resubmit, human)"
        resubmit, human = out
        by_id = {r["custom_id"]: r for r in resubmit}
        assert len(by_id) == len(resubmit), "each custom_id appears once in the resubmission"
        assert "ok-1" not in by_id, "never resubmit a success: you would pay twice and risk duplicate records"
        for cid in ("exp-1", "can-1", "ovl-1", "srv-1"):
            assert cid in by_id, f"{cid}: never ran or failed transiently: resubmit it unchanged"
            assert by_id[cid]["document"] == reqs[cid]["document"] and by_id[cid]["max_tokens"] == 1024, f"{cid}: resubmit UNCHANGED"
        chunks = split_lines(long_doc, 2_000)
        for cid in ("long-1", "long-2"):
            assert cid not in by_id, f"{cid}: resending a too-long document unchanged fails the same way: chunk it"
            got = [by_id.get(f"{cid}-c{i}", {}).get("document") for i in range(len(chunks))]
            assert got == chunks, f"{cid}: expected {len(chunks)} chunks {cid}-c0..-c{len(chunks) - 1} built with split_lines(document, max_chunk_chars)"
        assert by_id.get("trunc-1", {}).get("max_tokens") == 2048, "truncated: resubmit with max_tokens doubled (1024 -> 2048)"
        assert sorted(human) == ["bad-1", "ref-1"], f"refusals and non-transient request errors go to a person; got {sorted(human)}"
        expected_ids = {"exp-1", "can-1", "ovl-1", "srv-1", "trunc-1"} | {f"long-{k}-c{i}" for k in (1, 2) for i in range(len(chunks))}
        assert set(by_id) == expected_ids, f"unexpected resubmissions: {sorted(set(by_id) - expected_ids)}"
    _run_check(4, check)


check_ex4()

### Exercise 5 — SLA plan and cost per accepted document

Write two functions:

- `sla_plan(sla_hours, processing_hours=24, buffer_hours=2)` returns `{"window_hours": float, "retry_via": "batch" | "sync"}`. Prefer batch-priced retries when two batch cycles fit with a window of **at least 1 hour** (`sla − 2 × processing − buffer ≥ 1`). Here `"batch"` means **one** batch-priced retry round; any later round runs synchronously, as `batch_cycles=2` does in the practice notebook. Otherwise retries go synchronous and the window is `sla − processing − buffer`. If that is `≤ 0`, raise `ValueError`.
- `cost_per_accepted(usage, prices, n_accepted, field_reviews, review_cost=0.40)` where `usage = {"batch": (in_tokens, out_tokens), "sync": (in_tokens, out_tokens)}` (either key may be missing) and `prices = (input, output)` in USD per million tokens at **standard** rates. Batch tokens cost 50%; sync tokens cost 100%; each field review costs `review_cost`. Return total cost divided by `n_accepted`; raise `ValueError` if `n_accepted <= 0`.

Theory: [README 3.6](README.md#36-sla-math), [README 3.9](README.md#39-cost-comparison)

In [ ]:
def sla_plan(sla_hours: float, processing_hours: float = 24, buffer_hours: float = 2) -> dict:
    """Rules in the task above."""
    # TODO
    raise NotImplementedError


def cost_per_accepted(usage: dict, prices: tuple[float, float], n_accepted: int, field_reviews: int,
                      review_cost: float = 0.40) -> float:
    """Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        for sla, expected, why in [(30, {"window_hours": 4, "retry_via": "sync"}, "30 - 24 - 2 = 4; two cycles (50 h) do not fit"),
                                   (36, {"window_hours": 10, "retry_via": "sync"}, "36 - 24 - 2"),
                                   (54, {"window_hours": 4, "retry_via": "batch"}, "54 - 48 - 2 = 4: batch-priced retries fit"),
                                   (51, {"window_hours": 1, "retry_via": "batch"}, "51 - 48 - 2 = 1: the minimum batch window"),
                                   (50, {"window_hours": 24, "retry_via": "sync"},
                                    "50 - 48 - 2 = 0 leaves no batch window, so retries go sync: 50 - 24 - 2 = 24")]:
            got = sla_plan(sla)
            assert isinstance(got, dict) and got.get("retry_via") == expected["retry_via"] and \
                abs(float(got.get("window_hours", -1)) - expected["window_hours"]) < 1e-9, f"sla_plan({sla}) = {got}, expected {expected} ({why})"
        got = sla_plan(40, processing_hours=12)
        assert got.get("retry_via") == "batch" and abs(got["window_hours"] - 14) < 1e-9, "use the processing_hours you are given (40 - 24 - 2)"
        for sla in (26, 20):
            try:
                got = sla_plan(sla)
            except ValueError:
                continue
            raise AssertionError(f"sla_plan({sla}) returned {got}; a batch cannot meet it, raise ValueError")
        usage = {"batch": (1_000_000, 200_000), "sync": (100_000, 20_000)}
        got = cost_per_accepted(usage, (2.0, 10.0), n_accepted=90, field_reviews=20)
        assert isinstance(got, (int, float)) and abs(got - 10.4 / 90) < 1e-9, (
            f"expected ${10.4 / 90:.4f}: batch tokens $2.00 (50% of $4.00) + sync $0.40 + 20 reviews x $0.40 = $10.40, "
            f"divided by 90 ACCEPTED documents; got {got!r}")
        got = cost_per_accepted({"batch": (1_000_000, 200_000)}, (2.0, 10.0), n_accepted=100, field_reviews=0)
        assert abs(got - 0.02) < 1e-9, f"a missing 'sync' key means no sync tokens; expected $0.02, got {got!r}"
        try:
            cost_per_accepted(usage, (2.0, 10.0), n_accepted=0, field_reviews=5)
        except ValueError:
            pass
        else:
            raise AssertionError("n_accepted = 0: raise ValueError instead of dividing by zero")
    _run_check(5, check)


check_ex5()

### Exercise 6 — Thresholds per segment, and field-level routing

Write two functions:

- `pick_thresholds(records, target=0.97, min_n=20)`: `records` are labeled rows `{"doc_type", "field", "confidence", "correct"}`. For **each (doc_type, field)** pair, return the **lowest** confidence cut-off `t` such that the rows with `confidence >= t` reach `target` accuracy **and** number at least `min_n`; stop searching as soon as fewer than `min_n` rows remain. If no cut-off qualifies, the value is `None` (always review). Return a dict keyed by `(doc_type, field)`.
- `route_fields(extraction, thresholds)`: return the list of fields in `extraction["field_confidence"]` that need a reviewer: those with no threshold for `(extraction["doc_type"], field)` (missing or `None`), and those whose confidence is below the threshold.

Theory: [README 3.7](README.md#37-confidence-calibration-and-human-review-routing)

In [ ]:
def pick_thresholds(records: list[dict], target: float = 0.97, min_n: int = 20) -> dict:
    """Rules in the task above."""
    # TODO
    raise NotImplementedError


def route_fields(extraction: dict, thresholds: dict) -> list[str]:
    """Rules in the task above."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    def rows(doc_type, field, conf, n_correct, n_wrong):
        return ([{"doc_type": doc_type, "field": field, "confidence": conf, "correct": True}] * n_correct +
                [{"doc_type": doc_type, "field": field, "confidence": conf, "correct": False}] * n_wrong)

    def check():
        records = (rows("invoice", "total", 0.95, 100, 0) + rows("invoice", "total", 0.80, 35, 15) +
                   rows("invoice", "date", 0.99, 15, 0) + rows("invoice", "date", 0.60, 10, 0) +
                   rows("receipt", "vendor", 0.90, 24, 6) +
                   rows("credit_note", "total", 0.99, 10, 0))
        got = pick_thresholds(records)
        assert isinstance(got, dict), "return a dict keyed by (doc_type, field)"
        expected = {("invoice", "total"): 0.95, ("invoice", "date"): 0.60, ("receipt", "vendor"): None, ("credit_note", "total"): None}
        for key, want in expected.items():
            assert key in got, f"missing key {key}: return an entry (possibly None) for every (doc_type, field) in the records"
            have = got[key]
            if key == ("invoice", "total"):
                assert have == want, f"{key}: 0.80 and above is only 90% accurate; 0.95 and above is 100% with n=100, so {want}; got {have}"
            if key == ("invoice", "date"):
                assert have == want, (f"{key}: everything is correct and 25 rows >= 0.60, so the LOWEST cut-off 0.60 qualifies "
                                      f"(0.99 has only 15 rows < min_n); got {have}")
            if key == ("receipt", "vendor"):
                assert have is None, f"{key}: 80% accurate at every cut-off: no safe threshold, expected None; got {have}"
            if key == ("credit_note", "total"):
                assert have is None, f"{key}: only 10 labeled rows < min_n=20: too few to trust, expected None; got {have}"
        thresholds = {("invoice", "total"): 0.95, ("invoice", "date"): 0.60, ("invoice", "vendor"): None}
        ext = {"doc_type": "invoice", "field_confidence": {"total": 0.96, "date": 0.50, "vendor": 0.99, "po": 0.99}}
        flagged = route_fields(ext, thresholds)
        assert isinstance(flagged, list), "route_fields returns a list of field names"
        assert sorted(flagged) == ["date", "po", "vendor"], (
            f"expected ['date', 'po', 'vendor']: date is below its threshold, vendor has None (always review), po has no "
            f"calibrated threshold at all; total 0.96 >= 0.95 is auto-accepted. Got {sorted(flagged)}")
        ext2 = {"doc_type": "receipt", "field_confidence": {"total": 0.99}}
        assert route_fields(ext2, thresholds) == ["total"], "thresholds are per document type: an invoice threshold does not cover receipts"
    _run_check(6, check)


check_ex6()

## Part C — Architecture scenario

**Customs declarations for a freight forwarder.** A forwarder receives about **12,000 shipping documents per day** (commercial invoices, packing lists, certificates of origin, air waybills, and "misc" scans) in nine languages. For each, 20 fields go into the customs-declaration system. **Express** shipments (15% of volume) must be declared within **4 hours** of document arrival; **standard** shipments within **48 hours**. The compliance team can absorb about **1,500 field reviews per day**. Wrong HS codes and declared values cause fines; a wrong consignee phone number is an inconvenience. The client already has contracts with both Anthropic and OpenAI and asks you to recommend one stack, or a split.

Describe your design (15–25 lines):
1. Which documents go to a batch API and which to synchronous calls, with the SLA arithmetic (including retries).
2. The extraction contract: which fields are nullable, where you use enums with `"other"` + detail or `"unclear"`, and printed vs computed values.
3. Prompt and few-shot strategy across nine languages and five layouts, and how you choose it before the first full run.
4. Validation and retry: what is retried, what is not, and caps.
5. Batch mechanics: `custom_id` design, resubmission rules, oversized documents, and what differs between the two vendors' batch APIs that matters here.
6. Calibration and routing: segments, thresholds, priorities against the 1,500-review capacity, the audit sample.
7. How you will prove accuracy by segment before reducing review, and how you will compare the two vendors (and on what cost metric).

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **API split by SLA, with arithmetic.** Express (4 h) cannot use batches (no latency SLA, up to 24 h) → **synchronous** calls, with a smaller, cheaper model if the labeled sample supports it. Standard (48 h): submission window `W ≤ 48 − 24 − 2 = 22 h` with **synchronous** retries; batch-priced retries would need `W + 48 + 2 ≤ 48`, which is impossible, so retries go synchronous (or the client accepts a longer SLA). No "batch first, fall back to sync" for the express path.
- **Contract.** Nullable for anything a document can lack (consignee phone, origin certificate number, incoterm); enum + `"other"` + detail for document type, incoterm and currency; `"unclear"` where the source is ambiguous; declared value **as printed** plus a computed sum of line values and a `conflict_detected` flag; HS codes extracted as printed (validated by pattern and, ideally, against a reference list in code), never "completed" by the model; field-level confidence checked to 0–1 in code. Vendor rules respected (optional keys on Claude vs required + nullable on OpenAI; no `minimum`/`maximum` in Claude strict schemas).
- **Prompt choice before scale.** A labeled, **stratified** sample by language × document type; 2–4 targeted examples for the failure layouts (decimal commas, multi-page line tables, waybills vs invoices); per-segment before/after table; dry-run the exact request shape synchronously.
- **Validation and retry.** Schema + semantic checks (dates, HS-code pattern, sums, `"other"` without detail); retries send document + failed extraction + specific errors; absent information stays `null` and goes to review after at most one retry; never "must not be null".
- **Batch mechanics.** `custom_id` from the document's record key (hashed if it carries PII), `-cN` suffixes for chunks, stable across retries, ledger with batch IDs persisted at submit; resubmit only failures with the right modification (as is, chunked, larger output cap, feedback). Vendor differences that matter: OpenAI's one model per input file (a model split means two files), 50,000 requests and 200 MB per file vs Claude's 100,000 requests or 256 MB; OpenAI's output and error files vs Claude's single results stream; retention (29 days vs 30-day default file expiry); Claude's batch endpoint is not available on Bedrock, Vertex AI or Foundry; forced `tool_choice` restrictions on some Claude models.
- **Calibration and routing.** Thresholds per **language × document type × field** from labeled data with a minimum sample size; fields with no safe threshold always reviewed; queue priority = P(error) × cost (HS codes and declared values first), cut at 1,500/day; express items get queue priority because of their deadline; a stratified audit of auto-accepted output that is never skipped; reviewer corrections feed recalibration.
- **Evidence and vendor choice.** An accuracy matrix by segment and field (with n), compared with the aggregate, signed off before review is reduced; both vendors run on the same labeled sample; compared on **cost per accepted declaration** (tokens + retries + review load + fine risk), plus data-governance terms and platform fit. A split (for example one vendor for express sync and the other for standard batch) is acceptable if the numbers justify it and the adapters keep the pipeline vendor-neutral.

</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_ex = 6
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (10 + n_ex)
print(f"Quiz:       {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: tick the acceptance criteria in the README and write your recommendation" if overall >= 0.72
      else "RETRY: reread the README sections behind your ✗ items, then try again")

Next: tick the [acceptance criteria](README.md#acceptance-criteria), score yourself with the [rubric](README.md#rubric), then move on to [Capstone 4 — Coding agents for a team](../04-coding-agents-for-a-team/README.md).

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **B.** Required + nullable gives the model an honest way to say "not printed", which removes the pressure to invent. A prompt line alone (A) leaves the schema demanding a value. C is a brittle heuristic that also deletes genuine one-year expiries. D asks again for information that is not in the source, which tends to produce more confident fabrication.

**Q2:** **C.** `"other"` + a detail string keeps unanticipated cases without forcing them into a wrong bucket, and reviewing the detail values tells you which categories to add. A grows an unmanageable enum that will still be incomplete. B loses consistency for downstream reporting. D does not address the missing category at all.

**Q3:** **D.** The information is absent from the source, so `null` is the correct extraction. Retries (A, B) cannot create it and the "must not be null" feedback actively invites fabrication. C teaches the model to find VAT numbers where none exist. Whether those forms need a person is a business rule, not a model problem.

**Q4:** **A.** Strict schemas eliminate syntax errors, not semantic ones. A code-level check plus retry-with-specific-feedback fixes extraction mistakes, and a conflict flag preserves genuine errors in the source for a person. B misunderstands what strict mode guarantees. C hides real discrepancies (the source may be wrong). D addresses truncation, which is not the symptom.

**Q5:** **B.** `any` forces a tool call and lets the model pick the schema that fits the document. `auto` (A) may return text instead. Forcing one tool (C) misfiles receipts and contracts. `none` (D) forbids tool calls. (On models that reject forced tool use, the equivalent is `auto` with strict tools plus a code check, or a single JSON output schema with a type discriminator.)

**Q6:** **C.** Resubmit only failures, each with the fix that addresses its cause: expired requests never ran (unchanged), oversized documents need chunking (new IDs with a chunk suffix), and truncated outputs need a larger output cap. A repeats two deterministic failures. B reprocesses 1,940 successes. D sends fixable work to humans.

**Q7:** **B.** Worst case = W + 24 + 2 ≤ 40, so W ≤ 14 hours. A plans from the typical completion time instead of the 24-hour bound. C forgets the 2-hour buffer. D ignores that the SLA runs from arrival, not from submission.

**Q8:** **D.** A self-reported score is a ranking until a labeled set shows what each band means, per document type and field. A treats an uncalibrated number as a probability. B changes nothing measurable. C adds cost and still needs calibration.

**Q9:** **A.** A stratified random audit of auto-accepted output is the only unbiased measurement of its error rate, and it surfaces novel error patterns early. B only sees what is already flagged. C looks at the edge of the accepted set, not at confident errors. D measures through customer harm.

**Q10:** **C.** Each Batch API input file may contain requests for one model only, so routing to two models means two files and two batches. A does not exist (the model belongs in each line's `body`). B changes nothing about the rule. D is cosmetic.

</details>

<details><summary>Exercise 1 — Claim schemas: reference solution</summary>

```python
def _claim_props(provider: str) -> dict:
    def nullable(t: str) -> dict:
        return {"type": [t, "null"]}
    return {
        "claim_type": {"type": "string", "enum": ["water_damage", "fire", "theft", "other"]},
        "claim_type_detail": nullable("string"),
        "policy_number": nullable("string"),
        "incident_date": nullable("string"),
        "claimed_amount_cents": nullable("integer"),
        "police_report_ref": {"type": "string"} if provider == "claude" else nullable("string"),
        "items": {"type": "array", "items": {
            "type": "object",
            "properties": {"description": {"type": "string"}, "amount_cents": {"type": "integer"}},
            "required": ["description", "amount_cents"],
            "additionalProperties": False}},
        "conflict_detected": {"type": "boolean"},
        "field_confidence": {
            "type": "object",
            "properties": {f: {"type": "number"} for f in CONFIDENCE_FIELDS},
            "required": list(CONFIDENCE_FIELDS),
            "additionalProperties": False,
        },
    }


def build_claim_schemas() -> tuple[dict, dict]:
    schemas = []
    for provider in ("claude", "openai"):
        props = _claim_props(provider)
        required = [k for k in props if not (provider == "claude" and k == "police_report_ref")]
        schemas.append({"type": "object", "properties": props, "required": required, "additionalProperties": False})
    return schemas[0], schemas[1]
```
</details>

<details><summary>Exercise 2 — Semantic validator: reference solution</summary>

```python
def validate_claim(data: dict, attempt: int = 0) -> list[tuple[str, str, bool]]:
    try:
        claim = ClaimExtraction.model_validate(data)
    except ValidationError:
        return [("root", "schema", True)]
    issues = []
    if claim.incident_date is not None:
        try:
            ok = bool(ISO_DATE.match(claim.incident_date)) and date.fromisoformat(claim.incident_date) is not None
        except ValueError:
            ok = False
        if not ok:
            issues.append(("incident_date", "date_format", True))
    if claim.claim_type == "other" and not claim.claim_type_detail:
        issues.append(("claim_type_detail", "other_without_detail", True))
    if any(not 0 <= c <= 1 for c in claim.field_confidence.values()):
        issues.append(("field_confidence", "confidence_range", True))
    if claim.claimed_amount_cents is None:
        issues.append(("claimed_amount_cents", "missing_amount", attempt == 0))   # one honest re-read, then a human
    elif sum(i.amount_cents for i in claim.items) != claim.claimed_amount_cents and not claim.conflict_detected:
        issues.append(("items", "sum_mismatch", True))
    return issues
```
</details>

<details><summary>Exercise 3 — Retry request: reference solution</summary>

```python
def build_retry_request(custom_id: str, document: str, failed: dict, issues: list[tuple[str, str, bool]],
                        attempt: int, max_attempts: int = 2) -> dict | None:
    retryable = [(f, code) for f, code, ok in issues if ok]
    if not retryable or attempt + 1 >= max_attempts:
        return None                                        # nothing a retry can fix, or the budget is spent
    feedback = ("Your extraction failed validation:\n" + "\n".join(f"- {f}: {code}" for f, code in retryable)
                + "\nCorrect only these fields. If a value is not in the document, keep it null.")
    return {"custom_id": custom_id, "params": {
        "model": MODEL, "max_tokens": 1024,
        "output_config": {"format": {"type": "json_schema", "schema": CLAIM_SCHEMA_CLAUDE}},
        "messages": [{"role": "user", "content": document},
                     {"role": "assistant", "content": json.dumps(failed)},
                     {"role": "user", "content": feedback}]}}
```
</details>

<details><summary>Exercise 4 — Resubmission plan: reference solution</summary>

```python
def plan_resubmission(outcomes: list[dict], requests: dict[str, dict],
                      max_chunk_chars: int = 2_000) -> tuple[list[dict], list[str]]:
    resubmit, human = [], []
    for o in outcomes:
        cid, status = o["custom_id"], o["status"]
        req = requests[cid]
        text = f"{o.get('error_type') or ''} {o.get('error_message') or ''}"
        if status == "ok":
            continue
        if status in ("expired", "canceled") or (status == "errored" and o.get("error_type") in TRANSIENT):
            resubmit.append({"custom_id": cid, "document": req["document"], "max_tokens": req["max_tokens"]})
        elif status == "errored" and ("prompt is too long" in text or "context_length_exceeded" in text):
            for i, part in enumerate(split_lines(req["document"], max_chunk_chars)):
                resubmit.append({"custom_id": f"{cid}-c{i}", "document": part, "max_tokens": req["max_tokens"]})
        elif status == "truncated":
            resubmit.append({"custom_id": cid, "document": req["document"], "max_tokens": req["max_tokens"] * 2})
        else:                                               # refused, invalid request, anything unexpected
            human.append(cid)
    return resubmit, human
```
</details>

<details><summary>Exercise 5 — SLA plan and cost per accepted document: reference solution</summary>

```python
def sla_plan(sla_hours: float, processing_hours: float = 24, buffer_hours: float = 2) -> dict:
    batch_window = sla_hours - 2 * processing_hours - buffer_hours     # first pass AND retries as batches
    if batch_window >= 1:
        return {"window_hours": batch_window, "retry_via": "batch"}
    sync_window = sla_hours - processing_hours - buffer_hours          # only the first pass is a batch
    if sync_window <= 0:
        raise ValueError(f"a {sla_hours} h SLA cannot absorb a {processing_hours} h batch cycle plus buffer")
    return {"window_hours": sync_window, "retry_via": "sync"}


def cost_per_accepted(usage: dict, prices: tuple[float, float], n_accepted: int, field_reviews: int,
                      review_cost: float = 0.40) -> float:
    if n_accepted <= 0:
        raise ValueError("no accepted documents")
    p_in, p_out = prices
    b_in, b_out = usage.get("batch", (0, 0))
    s_in, s_out = usage.get("sync", (0, 0))
    tokens = (b_in * p_in + b_out * p_out) * 0.5 / 1e6 + (s_in * p_in + s_out * p_out) / 1e6
    return (tokens + field_reviews * review_cost) / n_accepted
```
</details>

<details><summary>Exercise 6 — Thresholds and routing: reference solution</summary>

```python
def pick_thresholds(records: list[dict], target: float = 0.97, min_n: int = 20) -> dict:
    groups = defaultdict(list)
    for r in records:
        groups[(r["doc_type"], r["field"])].append(r)
    out = {}
    for key, rows in groups.items():
        out[key] = None
        for t in sorted({r["confidence"] for r in rows}):
            above = [r["correct"] for r in rows if r["confidence"] >= t]
            if len(above) < min_n:
                break
            if sum(above) / len(above) >= target:
                out[key] = t
                break
    return out


def route_fields(extraction: dict, thresholds: dict) -> list[str]:
    flagged = []
    for f, c in extraction["field_confidence"].items():
        t = thresholds.get((extraction["doc_type"], f))
        if t is None or c < t:
            flagged.append(f)
    return flagged
```
</details>